# FER2013+ Multi-Backbone 7-Class Fine-Tuning & Validation-Optimized Fusion Pipeline
**Author:** Shahzaib Hussain | **Final Year Project (FYP)**  
**Dataset:** FER2013+ Cleaned 7-Class Ekman Benchmark (35,269 Facial Crops: 28,221 Train, 7,048 Test)  
**Affective Taxonomy (7 Classes):** `anger`, `disgust`, `fear`, `happiness`, `neutral`, `sadness`, `surprise`  
**Vision Backbones:** EfficientNet-B2 (9.1M), ResNet-50 (25.6M), Vision Transformer ViT-B/16 (86.6M)  
**Methodology:** 100% Parameter Pre-Trained Head Transfer, Class-Balanced Smoothed Cross-Entropy Loss, Cutout/Affine Data Augmentation, PyTorch AMP Mixed Precision, and Validation-Optimized Dirichlet Soft-Voting Probability Fusion.  
**Production Artifact:** Single unified PyTorch bundle (`outputs/fused_model.pt`) with real-time single-image inference.

In [ ]:
from IPython.display import display
# 1. Dependencies and environment verification
import os
import sys
import copy
import hashlib
import json
import random
import time
import warnings
import gc
from contextlib import nullcontext
from pathlib import Path
from typing import Dict, List, Tuple, Any, Optional

import numpy as np
import pandas as pd
import yaml
from PIL import Image, ImageFile
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.models as tv_models
import torchvision.transforms as T

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix, roc_auc_score, roc_curve

ImageFile.LOAD_TRUNCATED_IMAGES = True
warnings.filterwarnings("ignore", category=UserWarning)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision("high")
    gpu_name = torch.cuda.get_device_name(0)
    gpu_vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"[HARDWARE] GPU Active: {gpu_name} ({gpu_vram:.1f} GB VRAM) | CUDA: {torch.version.cuda}")
else:
    print("[HARDWARE] Running on CPU")


In [ ]:
# 2. Load configuration, seed all RNGs, and prepare directories
CONFIG_CANDIDATES = [
    Path.cwd() / "fine_tuning_pipeline.yaml",
    Path(r"C:\Users\Shahzaib Hussain\Desktop\Shahzaib Hussain\FYP\Voice and Facial recognition\NewNotebook of FineTuning\fine_tuning_pipeline.yaml")
]

config_path = None
for candidate in CONFIG_CANDIDATES:
    if candidate.exists():
        config_path = candidate
        break

if config_path is None:
    raise FileNotFoundError(f"Configuration file not found in candidates: {CONFIG_CANDIDATES}")

with open(config_path, "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

PROJECT_DIR = config_path.parent

def resolve_path(value, base=PROJECT_DIR):
    if value is None: return None
    p = Path(value)
    return p if p.is_absolute() else (base / p).resolve()

OUTPUT_DIR = resolve_path(cfg["project"].get("output_dir", "./outputs"))
CHECKPOINT_DIR = resolve_path(cfg["project"].get("checkpoint_dir", "./checkpoints"))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

SEED = int(cfg["project"].get("seed", 42))
random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

class_names = [str(x) for x in cfg["labels"]["class_names"]]
print(f"[CONFIG] Loaded configuration from: {config_path.name}")
print(f"[CONFIG] Active 7 Classes : {class_names}")
print(f"[CONFIG] Outputs Directory: {OUTPUT_DIR}")
print(f"[CONFIG] Checkpoints Dir  : {CHECKPOINT_DIR}")


In [ ]:
# 3. Dataset inventory and fast auditing with caching
DATA_ROOT = resolve_path(cfg["data"]["root"])
if not DATA_ROOT.exists():
    raise FileNotFoundError(f"Dataset root does not exist: {DATA_ROOT}")

# Dynamically resolve dataset root if nested
active_root = DATA_ROOT
if not (active_root / "train").exists() and (active_root / "fer2013" / "train").exists():
    active_root = active_root / "fer2013"

inv_path = OUTPUT_DIR / "dataset_inventory.csv"
EXTENSIONS = {str(x).lower() for x in cfg["data"].get("extensions", [".jpg", ".jpeg", ".png", ".bmp", ".webp"])}

# Check cache validity
cache_valid = False
if inv_path.exists():
    try:
        manifest = pd.read_csv(inv_path)
        manifest_classes = sorted(manifest["label"].unique().tolist())
        first_exists = Path(manifest.iloc[0]["path"]).exists() if len(manifest) > 0 else False
        if manifest_classes == sorted(class_names) and len(manifest) > 30000 and first_exists:
            cache_valid = True
            print(f"[CACHE] Loaded verified 7-class dataset inventory: {len(manifest):,} files from: {active_root}")
    except Exception:
        cache_valid = False

if not cache_valid:
    print(f"[SCAN] Scanning dataset directories under: {active_root} ...")
    rows = []
    for split_name in ["train", "test"]:
        split_dir = active_root / split_name
        if not split_dir.exists():
            continue
        for class_dir in sorted(split_dir.iterdir()):
            class_name = class_dir.name
            if class_name not in class_names:
                continue
            for p in class_dir.iterdir():
                if p.is_file() and p.suffix.lower() in EXTENSIONS:
                    rows.append({
                        "path": str(p),
                        "split": split_name,
                        "label": class_name,
                        "bytes": p.stat().st_size
                    })
    manifest = pd.DataFrame(rows)
    manifest.to_csv(inv_path, index=False)
    print(f"[SCAN] Generated and cached 7-class inventory: {len(manifest):,} files.")

inventory_summary = manifest.groupby(["split", "label"]).size().unstack(fill_value=0)[class_names]
display(inventory_summary)


In [ ]:
# 4. Exploratory Data Analysis (EDA) visualizations
fig, axes = plt.subplots(1, 3, figsize=(21, 5.5))

# Panel 1: Partition distribution
split_dist = manifest.groupby(["split", "label"]).size().unstack(fill_value=0).reindex(columns=class_names, fill_value=0)
split_dist.plot(kind="bar", ax=axes[0], colormap="Set2", edgecolor="black", linewidth=0.5)
axes[0].set_title("Class Frequency by Official Partition", fontsize=12, fontweight="bold")
axes[0].set_ylabel("Number of Facial Images", fontweight="bold")
axes[0].set_xlabel("Partition", fontweight="bold")
axes[0].tick_params(axis="x", rotation=0)
axes[0].grid(axis="y", alpha=0.3)
axes[0].legend(frameon=True, fontsize=9)

# Panel 2: File size distribution
file_sizes_kb = manifest["bytes"] / 1024.0
sns.histplot(file_sizes_kb, bins=35, ax=axes[1], color="#0284c7", kde=True, edgecolor="black")
axes[1].set_title("File Size Distribution (KB)", fontsize=12, fontweight="bold")
axes[1].set_xlabel("File Size (KB)", fontweight="bold")
axes[1].grid(alpha=0.3)

# Panel 3: Imbalance breakdown
class_counts = manifest["label"].value_counts()[class_names]
max_count = class_counts.max()
imbalance_ratios = max_count / class_counts

colors = sns.color_palette("viridis", len(class_names))
bars = axes[2].barh(class_names, class_counts, color=colors, edgecolor="black", linewidth=0.5)
axes[2].set_title(f"Class Distribution ({len(manifest):,} Images Total)", fontsize=12, fontweight="bold")
axes[2].set_xlabel("Sample Count", fontweight="bold")
axes[2].grid(axis="x", alpha=0.3)

for bar in bars:
    w = bar.get_width()
    axes[2].text(w + 120, bar.get_y() + bar.get_height()/2, f"{w:,}", va="center", fontsize=9, fontweight="bold")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "eda_dataset_overview.png", dpi=200)
plt.show()
plt.close()


In [ ]:
# 5. Build official train/test records and stratified validation split
train_all = manifest[manifest.split.str.lower() == "train"].copy().reset_index(drop=True)
test_records = manifest[manifest.split.str.lower() == "test"].copy().reset_index(drop=True)

label_to_index = {name: i for i, name in enumerate(class_names)}
train_all["target"] = train_all["label"].map(label_to_index)
test_records["target"] = test_records["label"].map(label_to_index)

assert not train_all["target"].isna().any(), "Detected unknown labels in training partition"
assert not test_records["target"].isna().any(), "Detected unknown labels in test partition"

train_all["target"] = train_all["target"].astype(int)
test_records["target"] = test_records["target"].astype(int)

# Stratified Validation Split (15% of train partition)
val_fraction = float(cfg["data"].get("validation_fraction", 0.15))
sss = StratifiedShuffleSplit(n_splits=1, test_size=val_fraction, random_state=SEED)
train_idx, val_idx = next(sss.split(train_all, train_all["target"]))

train_records = train_all.iloc[train_idx].copy().reset_index(drop=True)
val_records = train_all.iloc[val_idx].copy().reset_index(drop=True)

partition_summary = pd.DataFrame({
    "Train": train_records["label"].value_counts()[class_names],
    "Validation": val_records["label"].value_counts()[class_names],
    "Test": test_records["label"].value_counts()[class_names]
})
partition_summary["Total"] = partition_summary.sum(axis=1)
print(f"[DATASET] Split completed across {len(class_names)} classes:")
display(partition_summary)


In [ ]:
# 6. Advanced data transforms and PyTorch DataLoaders
image_size = int(cfg["data"].get("image_size", 224))
mean, std = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

# Training augmentation: spatial rotation, affine jitter, color adjustment, and subtle random erasing
train_transform = T.Compose([
    T.Resize((image_size, image_size)),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomRotation(degrees=12),
    T.RandomAffine(degrees=10, translate=(0.04, 0.04), scale=(0.96, 1.04)),
    T.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15),
    T.ToTensor(),
    T.Normalize(mean, std),
    T.RandomErasing(p=0.10, scale=(0.02, 0.15), value="random")
])

# Evaluation transform: deterministic resize and standard normalization
eval_transform = T.Compose([
    T.Resize((image_size, image_size)),
    T.ToTensor(),
    T.Normalize(mean, std)
])

# High-Performance Dataset using pre-extracted lists to eliminate pandas indexing overhead
class FastEmotionDataset(Dataset):
    def __init__(self, df: pd.DataFrame, transform):
        self.paths = df["path"].to_list()
        self.targets = df["target"].to_numpy(dtype=np.int64)
        self.transform = transform
        
    def __len__(self) -> int:
        return len(self.paths)
        
    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, int]:
        img = Image.open(self.paths[idx]).convert("RGB")
        return self.transform(img), int(self.targets[idx])

train_ds = FastEmotionDataset(train_records, train_transform)
val_ds = FastEmotionDataset(val_records, eval_transform)
test_ds = FastEmotionDataset(test_records, eval_transform)

batch_size = int(cfg["training"].get("batch_size", 64))
num_workers = int(cfg["training"].get("num_workers", 0))
pin_memory = bool(cfg["training"].get("pin_memory", True)) and device.type == "cuda"

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=pin_memory)
val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=pin_memory)
test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=pin_memory)

print(f"[DATALOADERS] Initialized (Batch Size: {batch_size}, Pin Memory: {pin_memory}, Num Workers: {num_workers})")
print(f"[DATALOADERS] Training Batches: {len(train_loader)} | Validation Batches: {len(val_loader)} | Test Batches: {len(test_loader)}")


In [ ]:
# 7. Model discovery, construction, and Smart Checkpoint Head Transfer
PATTERNS = {
    "efficientnet_b2": ["best_model efficient_b2.pt", "*efficient*b2*.pt"],
    "resnet50": ["best_model resnet50.pt", "*resnet*50*.pt"],
    "vit_b_16": ["best_model vit b_16.pt", "*vit*b*16*.pt"]
}

def first_existing(patterns):
    for pattern in patterns:
        found = sorted(PROJECT_DIR.glob(pattern))
        if found: return found[0]
    return None

checkpoint_paths = {name: first_existing(patterns) for name, patterns in PATTERNS.items()}
print("Discovered Pre-Trained Checkpoints:", {k: v.name if v else None for k, v in checkpoint_paths.items()})

LABEL_MAPPING = {
    "angry": "anger",
    "anger": "anger",
    "disgust": "disgust",
    "fear": "fear",
    "happy": "happiness",
    "happiness": "happiness",
    "neutral": "neutral",
    "sad": "sadness",
    "sadness": "sadness",
    "surprise": "surprise"
}

def build_model(name: str, num_classes: int = len(class_names)) -> nn.Module:
    if name == "efficientnet_b2":
        model = tv_models.efficientnet_b2(weights=None)
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
    elif name == "resnet50":
        model = tv_models.resnet50(weights=None)
        model.fc = nn.Linear(model.fc.in_features, num_classes)
    elif name == "vit_b_16":
        model = tv_models.vit_b_16(weights=None)
        model.heads.head = nn.Linear(model.heads.head.in_features, num_classes)
    else:
        raise ValueError(f"Unknown backbone: {name}")
    return model

def extract_state_dict(payload):
    if isinstance(payload, nn.Module): return payload.state_dict()
    if isinstance(payload, dict):
        for key in ("state_dict", "model_state_dict", "model", "net"):
            if isinstance(payload.get(key), dict): return payload[key]
        if all(isinstance(k, str) for k in payload): return payload
    raise TypeError("Unsupported checkpoint format")

def load_checkpoint_smart(model: nn.Module, path: Optional[Path], model_name: str):
    if not path or not path.exists():
        print(f"[{model_name}] No checkpoint found; initializing with standard weights.")
        return
        
    payload = torch.load(path, map_location="cpu", weights_only=False)
    state = extract_state_dict(payload)
    state = {k.replace("module.", "", 1): v for k, v in state.items()}
    current = model.state_dict()
    
    # 1. Load all compatible backbone parameters
    compatible = {k: v for k, v in state.items() if k in current and current[k].shape == v.shape}
    model.load_state_dict(compatible, strict=False)
    print(f"[{model_name}] Loaded {len(compatible):,} compatible backbone parameter tensors from {path.name}.")
    
    # 2. Checkpoint Head Transfer
    old_labels = payload.get("labels", ["angry", "disgust", "fear", "happy", "sad", "surprise", "neutral"])
    new_idx_map = {name: i for i, name in enumerate(class_names)}
    
    head_w_key, head_b_key, head_layer = None, None, None
    if model_name == "efficientnet_b2":
        head_w_key, head_b_key = "classifier.1.weight", "classifier.1.bias"
        head_layer = model.classifier[1]
    elif model_name == "resnet50":
        head_w_key, head_b_key = "fc.weight", "fc.bias"
        head_layer = model.fc
    elif model_name == "vit_b_16":
        head_w_key, head_b_key = "heads.head.weight", "heads.head.bias"
        head_layer = model.heads.head
        
    if head_w_key in state and head_b_key in state and head_layer is not None:
        old_w, old_b = state[head_w_key], state[head_b_key]
        new_w, new_b = head_layer.weight.data.clone(), head_layer.bias.data.clone()
        transferred = 0
        for old_idx, old_name in enumerate(old_labels):
            mapped = LABEL_MAPPING.get(old_name.lower())
            if mapped in new_idx_map:
                n_idx = new_idx_map[mapped]
                new_w[n_idx] = old_w[old_idx]
                new_b[n_idx] = old_b[old_idx]
                transferred += 1
        head_layer.weight.data.copy_(new_w)
        head_layer.bias.data.copy_(new_b)
        print(f"[{model_name}] 100% Parameter Head Transfer: mapped {transferred}/{len(class_names)} classes directly!")


In [ ]:
# 8. Training utilities, class-balanced loss, and prediction routines
def get_class_weights(frame: pd.DataFrame, beta: float = 0.60) -> torch.Tensor:
    counts = np.bincount(frame.target.to_numpy(), minlength=len(class_names)).astype(np.float32)
    max_c = counts.max()
    weights = (max_c / np.maximum(counts, 1.0)) ** beta
    weights = weights / weights.mean()
    return torch.tensor(weights, dtype=torch.float32, device=device)

def run_epoch(model: nn.Module, loader: DataLoader, criterion: nn.Module, optimizer: Optional[torch.optim.Optimizer] = None, scaler: Optional[torch.amp.GradScaler] = None) -> Dict[str, float]:
    training = optimizer is not None
    model.train(training)
    total_loss = 0.0
    truth, pred = [], []
    amp = bool(cfg["training"].get("mixed_precision", True)) and device.type == "cuda"
    
    for images, targets in loader:
        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)
        
        if training:
            optimizer.zero_grad(set_to_none=True)
            
        with (torch.amp.autocast("cuda", dtype=torch.float16) if amp else nullcontext()):
            logits = model(images)
            loss = criterion(logits, targets)
            
        if training:
            if amp and scaler is not None:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), float(cfg["training"].get("gradient_clip_norm", 1.0)))
                scaler.step(optimizer)
                scaler.update()
            else:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), float(cfg["training"].get("gradient_clip_norm", 1.0)))
                optimizer.step()
                
        total_loss += loss.item() * len(targets)
        truth.extend(targets.cpu().numpy())
        pred.extend(logits.argmax(1).detach().cpu().numpy())
        
    return {
        "loss": total_loss / max(len(loader.dataset), 1),
        "accuracy": accuracy_score(truth, pred),
        "macro_f1": f1_score(truth, pred, average="macro", zero_division=0)
    }

@torch.no_grad()
def predict(model: nn.Module, loader: DataLoader) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    model.eval()
    truth, pred, probs = [], [], []
    amp = bool(cfg["training"].get("mixed_precision", True)) and device.type == "cuda"
    for images, targets in loader:
        with (torch.amp.autocast("cuda", dtype=torch.float16) if amp else nullcontext()):
            outputs = model(images.to(device, non_blocking=True))
            probabilities = outputs if model.__class__.__name__ == "ProbabilityFusionModel" else torch.softmax(outputs, dim=1)
        probs.extend(probabilities.cpu().numpy().astype(np.float32))
        pred.extend(probabilities.argmax(1).cpu().numpy())
        truth.extend(targets.numpy())
    return np.array(truth), np.array(pred), np.array(probs)

def train_one(model_name: str) -> Tuple[Any, ...]:
    best_ckpt_file = CHECKPOINT_DIR / f"{model_name}_best.pt"
    history_file = OUTPUT_DIR / f"{model_name}_training_history.csv"
    
    # Check if a compatible checkpoint already exists
    if best_ckpt_file.exists() and history_file.exists():
        ck = torch.load(best_ckpt_file, map_location=device, weights_only=False)
        ck_classes = ck.get("class_names")
        if ck_classes == class_names:
            print(f"[{model_name}] Found compatible 7-class checkpoint ({best_ckpt_file.name}). Loading...")
            model = build_model(model_name).to(device)
            state = ck.get("state_dict", ck.get("model_state_dict", ck))
            model.load_state_dict(state)
            history = pd.read_csv(history_file)
            val_true, val_pred, val_probs = predict(model, val_loader)
            test_true, test_pred, test_probs = predict(model, test_loader)
            test_acc = accuracy_score(test_true, test_pred)
            test_f1 = f1_score(test_true, test_pred, average="macro", zero_division=0)
            print(f"[{model_name} LOADED] Test Accuracy: {test_acc*100:.2f}% | Test Macro-F1: {test_f1*100:.2f}%")
            return model, history, val_true, val_pred, val_probs, test_true, test_pred, test_probs
        else:
            print(f"[{model_name}] Checkpoint found but class list differs. Initiating fresh 7-class fine-tuning...")

    print(f"\n{'='*65}\n[TRAINING] Starting 7-Class Fine-Tuning: {model_name}\n{'='*65}")
    model = build_model(model_name).to(device)
    load_checkpoint_smart(model, checkpoint_paths.get(model_name), model_name)
    
    weights = get_class_weights(train_records, beta=float(cfg["training"].get("class_weight_beta", 0.60))) if cfg["training"].get("class_weights") == "balanced" else None
    criterion = nn.CrossEntropyLoss(weight=weights, label_smoothing=float(cfg["training"].get("label_smoothing", 0.05)))
    
    backbone, head = [], []
    for name, p in model.named_parameters():
        (head if any(x in name for x in ("classifier", "fc", "heads")) else backbone).append(p)
        
    base_lr = float(cfg["training"].get("learning_rate", 1e-4))
    head_lr = float(cfg["training"].get("head_learning_rate", 5e-4))
    if model_name == "vit_b_16":
        vit_factor = float(cfg["training"].get("vit_lr_factor", 0.35))
        base_lr *= vit_factor
        head_lr *= 0.50
        
    optimizer = torch.optim.AdamW([
        {"params": backbone, "lr": base_lr},
        {"params": head, "lr": head_lr}
    ], weight_decay=float(cfg["training"].get("weight_decay", 1e-4)))
    
    total_epochs = int(cfg["training"].get("epochs", 12))
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_epochs, eta_min=1e-6)
    scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda" and bool(cfg["training"].get("mixed_precision", True)))
    
    history = []
    best_state = None
    best_macro_f1 = -1.0
    stale = 0
    patience = int(cfg["training"].get("early_stopping_patience", 4))
    freeze_epochs = int(cfg["model"].get("freeze_backbone_epochs", 1))
    
    t_start = time.time()
    for epoch in range(total_epochs):
        t_ep = time.time()
        freeze = epoch < freeze_epochs
        for p in backbone: p.requires_grad = not freeze
            
        tr = run_epoch(model, train_loader, criterion, optimizer, scaler)
        va = run_epoch(model, val_loader, criterion)
        scheduler.step()
        
        row = {"epoch": epoch + 1, **{f"train_{k}": v for k, v in tr.items()}, **{f"val_{k}": v for k, v in va.items()}}
        history.append(row)
        ep_sec = time.time() - t_ep
        print(f"[{model_name}] Ep {epoch+1:02d}/{total_epochs:02d} ({ep_sec:.1f}s) | Train Acc: {tr['accuracy']*100:.2f}%, F1: {tr['macro_f1']*100:.2f}% | Val Acc: {va['accuracy']*100:.2f}%, F1: {va['macro_f1']*100:.2f}% (Loss: {va['loss']:.4f})")
        
        if va["macro_f1"] > best_macro_f1:
            best_macro_f1 = va["macro_f1"]
            best_state = copy.deepcopy(model.state_dict())
            stale = 0
            torch.save({
                "model_name": model_name,
                "class_names": class_names,
                "state_dict": best_state,
                "config": cfg,
                "val_macro_f1": best_macro_f1
            }, best_ckpt_file)
        else:
            stale += 1
            if stale >= patience:
                print(f"[EARLY STOPPING] {model_name} triggered early stopping at epoch {epoch+1}.")
                break
                
    total_time = time.time() - t_start
    print(f"[COMPLETED] {model_name} trained in {total_time/60:.2f} mins. Best Val Macro-F1: {best_macro_f1*100:.2f}%")
    model.load_state_dict(best_state)
    val_true, val_pred, val_probs = predict(model, val_loader)
    test_true, test_pred, test_probs = predict(model, test_loader)
    test_acc = accuracy_score(test_true, test_pred)
    test_f1 = f1_score(test_true, test_pred, average="macro", zero_division=0)
    print(f"[{model_name} HELD-OUT TEST] Accuracy: {test_acc*100:.2f}% | Macro-F1: {test_f1*100:.2f}%")
    
    return model, pd.DataFrame(history), val_true, val_pred, val_probs, test_true, test_pred, test_probs


In [ ]:
# 9. Fine-tune all three backbones sequentially
model_results = {}
active_models = cfg["model"].get("active_backbones", ["efficientnet_b2", "resnet50", "vit_b_16"])

for model_name in active_models:
    model_results[model_name] = train_one(model_name)
    history = model_results[model_name][1]
    history.to_csv(OUTPUT_DIR / f"{model_name}_training_history.csv", index=False)
    released = list(model_results[model_name])
    released[0] = None  # release in-memory model to keep VRAM clean
    model_results[model_name] = tuple(released)
    
    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()

print("\n[FINE-TUNING COMPLETE] All backbones finished fine-tuning!")


In [ ]:
# 10. Individual model reports, complete metrics, curves, ROC, and percentage confusion matrices
individual_metrics = []
fig, axes = plt.subplots(1, 3, figsize=(21, 5.5), constrained_layout=True)

for model_name, result in model_results.items():
    history, val_true, val_pred, val_probs, test_true, test_pred, test_probs = result[1:]
    epochs = history["epoch"].to_numpy()
    report = classification_report(test_true, test_pred, target_names=class_names, output_dict=True, zero_division=0)
    
    try:
        test_roc_auc = roc_auc_score(test_true, test_probs, multi_class="ovr", average="macro", labels=np.arange(len(class_names)))
    except ValueError:
        test_roc_auc = float("nan")
        
    metrics = {
        "model": model_name,
        "train_accuracy_final": float(history["train_accuracy"].iloc[-1]),
        "validation_accuracy_best": float(history["val_accuracy"].max()),
        "test_accuracy": float(accuracy_score(test_true, test_pred)),
        "train_loss_final": float(history["train_loss"].iloc[-1]),
        "validation_loss_best": float(history["val_loss"].min()),
        "test_macro_f1": float(f1_score(test_true, test_pred, average="macro", zero_division=0)),
        "test_micro_f1": float(f1_score(test_true, test_pred, average="micro", zero_division=0)),
        "test_weighted_f1": float(f1_score(test_true, test_pred, average="weighted", zero_division=0)),
        "test_roc_auc": float(test_roc_auc)
    }
    for c in class_names:
        metrics[f"f1_{c}"] = float(report[c]["f1-score"])
    individual_metrics.append(metrics)
    
    with open(OUTPUT_DIR / f"{model_name}_metrics.json", "w") as f:
        json.dump({"metrics": metrics, "report": report}, f, indent=2)
        
    # Plot training/validation trajectories
    axes[0].plot(epochs, history["train_loss"], label=f"{model_name} Train Loss", linestyle="--")
    axes[0].plot(epochs, history["val_loss"], label=f"{model_name} Val Loss")
    axes[1].plot(epochs, history["train_accuracy"] * 100, label=f"{model_name} Train Acc", linestyle="--")
    axes[1].plot(epochs, history["val_accuracy"] * 100, label=f"{model_name} Val Acc")
    axes[2].plot(epochs, history["train_macro_f1"] * 100, label=f"{model_name} Train F1", linestyle="--")
    axes[2].plot(epochs, history["val_macro_f1"] * 100, label=f"{model_name} Val F1")

    # Plot & Save Dual-Annotated Confusion Matrix
    cm = confusion_matrix(test_true, test_pred)
    cm_norm = cm.astype("float") / cm.sum(axis=1)[:, np.newaxis]
    annot = np.empty_like(cm, dtype=object)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            annot[i, j] = f"{cm_norm[i, j]*100:.1f}%\n({cm[i, j]})"
            
    fig_cm, ax_cm = plt.subplots(figsize=(8.5, 7.5))
    sns.heatmap(cm_norm * 100, annot=annot, fmt="", cmap="Blues", cbar=True,
                xticklabels=class_names, yticklabels=class_names, ax=ax_cm, annot_kws={"fontsize": 9, "fontweight": "bold"})
    ax_cm.set_title(f"{model_name.upper()} Held-Out Test Confusion Matrix\nTest Acc: {metrics['test_accuracy']*100:.2f}% | Macro F1: {metrics['test_macro_f1']*100:.2f}%", fontweight="bold", pad=12)
    ax_cm.set_xlabel("Predicted Emotion", fontweight="bold")
    ax_cm.set_ylabel("True Emotion", fontweight="bold")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f"{model_name}_confusion_matrix_percentage.png", dpi=200)
    plt.close()

    # Plot & Save Multiclass ROC Curves
    plt.figure(figsize=(9, 7))
    for idx_c, c_name in enumerate(class_names):
        y_binary = (test_true == idx_c).astype(int)
        if len(np.unique(y_binary)) > 1:
            fpr, tpr, _ = roc_curve(y_binary, test_probs[:, idx_c])
            auc_val = roc_auc_score(y_binary, test_probs[:, idx_c])
            plt.plot(fpr, tpr, label=f"{c_name.capitalize()} (AUC = {auc_val:.3f})")
    plt.plot([0, 1], [0, 1], "k--", alpha=0.5)
    plt.title(f"{model_name.upper()} One-vs-Rest ROC Curves (Macro AUC = {metrics['test_roc_auc']:.3f})", fontweight="bold")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.legend(loc="lower right")
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f"{model_name}_roc_curves.png", dpi=200)
    plt.close()

axes[0].set_title("Cross-Entropy Loss Trajectory", fontweight="bold"); axes[0].grid(alpha=0.3); axes[0].legend(fontsize=8)
axes[1].set_title("Classification Accuracy (%)", fontweight="bold"); axes[1].grid(alpha=0.3); axes[1].legend(fontsize=8)
axes[2].set_title("Macro F1-Score (%)", fontweight="bold"); axes[2].grid(alpha=0.3); axes[2].legend(fontsize=8)
plt.savefig(OUTPUT_DIR / "all_models_training_validation_metrics.png", dpi=200)
plt.show()
plt.close()

individual_df = pd.DataFrame(individual_metrics)
display(individual_df.round(4))
individual_df.to_csv(OUTPUT_DIR / "individual_model_comparison.csv", index=False)


In [ ]:
# 11. Validation-Optimized Dirichlet Probability Fusion
class ProbabilityFusionModel(nn.Module):
    def __init__(self, models: List[nn.Module], weights: Any):
        super().__init__()
        self.models = nn.ModuleList(models)
        self.register_buffer("weights", torch.tensor(weights, dtype=torch.float32))
        
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        probabilities = [torch.softmax(m(x), dim=1) for m in self.models]
        normalized_w = self.weights / self.weights.sum()
        return sum(w * p for w, p in zip(normalized_w, probabilities))

def load_best_model(m_name: str) -> nn.Module:
    m = build_model(m_name).to(device)
    ck = torch.load(CHECKPOINT_DIR / f"{m_name}_best.pt", map_location=device, weights_only=False)
    state = ck.get("state_dict", ck.get("model_state_dict", ck))
    m.load_state_dict(state)
    m.eval()
    return m

model_names = list(model_results.keys())
val_probs_dict = {name: model_results[name][4] for name in model_names}
val_true_labels = model_results[model_names[0]][2]

# Search for optimal Dirichlet fusion weights on validation split
print("[FUSION] Searching for optimal Dirichlet weights on validation partition...")
best_weights, best_val_f1 = None, -1.0
grid_weights = np.random.dirichlet(np.ones(len(model_names)), size=2500)
grid_weights = np.vstack([grid_weights, np.eye(len(model_names)), np.ones((1, len(model_names))) / len(model_names)])

for candidate in grid_weights:
    fused_val = sum(w * val_probs_dict[m] for w, m in zip(candidate, model_names))
    score = f1_score(val_true_labels, fused_val.argmax(1), average="macro", zero_division=0)
    if score > best_val_f1:
        best_val_f1, best_weights = score, candidate

weights = torch.tensor(best_weights / best_weights.sum(), dtype=torch.float32)
weight_summary = {name: round(float(w), 4) for name, w in zip(model_names, weights.tolist())}
print(f"[FUSION WINNER] Best Val Macro F1: {best_val_f1*100:.2f}% | Optimal Weights: {weight_summary}")

# Build fused model and evaluate on held-out test split
fused_model = ProbabilityFusionModel([load_best_model(name) for name in model_names], weights).to(device)
test_true_fused, test_pred_fused, test_probs_fused = predict(fused_model, test_loader)

fused_report = classification_report(test_true_fused, test_pred_fused, target_names=class_names, output_dict=True, zero_division=0)
fused_roc_auc = roc_auc_score(test_true_fused, test_probs_fused, multi_class="ovr", average="macro", labels=np.arange(len(class_names)))

fused_metrics = {
    "model": "fused_champion_ensemble",
    "train_accuracy_final": float(np.mean([m["train_accuracy_final"] for m in individual_metrics])),
    "validation_accuracy_best": float(best_val_f1),
    "test_accuracy": float(accuracy_score(test_true_fused, test_pred_fused)),
    "train_loss_final": float(np.mean([m["train_loss_final"] for m in individual_metrics])),
    "validation_loss_best": float(np.mean([m["validation_loss_best"] for m in individual_metrics])),
    "test_macro_f1": float(f1_score(test_true_fused, test_pred_fused, average="macro", zero_division=0)),
    "test_micro_f1": float(f1_score(test_true_fused, test_pred_fused, average="micro", zero_division=0)),
    "test_weighted_f1": float(f1_score(test_true_fused, test_pred_fused, average="weighted", zero_division=0)),
    "test_roc_auc": float(fused_roc_auc)
}
for c in class_names:
    fused_metrics[f"f1_{c}"] = float(fused_report[c]["f1-score"])

with open(OUTPUT_DIR / "fused_model_metrics.json", "w") as f:
    json.dump({"metrics": fused_metrics, "weights": weight_summary, "report": fused_report}, f, indent=2)

# Dual-Annotated Percentage & Count Confusion Matrix
cm_fused = confusion_matrix(test_true_fused, test_pred_fused)
cm_fused_norm = cm_fused.astype("float") / cm_fused.sum(axis=1)[:, np.newaxis]
annot_fused = np.empty_like(cm_fused, dtype=object)
for i in range(cm_fused.shape[0]):
    for j in range(cm_fused.shape[1]):
        annot_fused[i, j] = f"{cm_fused_norm[i, j]*100:.1f}%\n({cm_fused[i, j]})"

fig, ax = plt.subplots(figsize=(9, 8))
sns.heatmap(cm_fused_norm * 100, annot=annot_fused, fmt="", cmap="Blues", cbar=True,
            xticklabels=class_names, yticklabels=class_names, ax=ax, annot_kws={"fontsize": 10, "fontweight": "bold"})
ax.set_title(f"CHAMPION FUSED ENSEMBLE: Held-Out Test Confusion Matrix\nTest Acc: {fused_metrics['test_accuracy']*100:.2f}% | Macro F1: {fused_metrics['test_macro_f1']*100:.2f}% | Weighted F1: {fused_metrics['test_weighted_f1']*100:.2f}%", fontsize=11, fontweight="bold", pad=12)
ax.set_xlabel("Predicted Emotion", fontweight="bold")
ax.set_ylabel("True Emotion", fontweight="bold")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "fused_model_confusion_matrix_percentage.png", dpi=200)
plt.show()
plt.close()


In [ ]:
# 12. Final comparison, save fused_model.pt, reload, and verify inference
comparison = pd.concat([individual_df, pd.DataFrame([fused_metrics])], ignore_index=True)
display(comparison)
comparison.to_csv(OUTPUT_DIR / "complete_model_comparison.csv", index=False)

plt.figure(figsize=(14, 6))
comparison.set_index("model")[["test_accuracy", "test_macro_f1", "test_micro_f1", "test_weighted_f1"]].plot(kind="bar", figsize=(14, 6), ylim=(0, 1.05), title="Individual Models vs. Champion Fused Ensemble")
plt.xticks(rotation=15, fontweight="bold")
plt.ylabel("Score")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "model_and_fusion_comparison.png", dpi=200)
plt.show()
plt.close()

final_model_path = resolve_path(cfg["evaluation"].get("final_model_path", "./outputs/fused_model.pt"))
final_payload = {
    "model_type": "ProbabilityFusionModel",
    "model_names": model_names,
    "class_names": class_names,
    "num_classes": len(class_names),
    "fusion_weights": weights.tolist(),
    "image_size": image_size,
    "mean": mean,
    "std": std,
    "state_dict": fused_model.state_dict(),
    "component_state_dicts": [model.state_dict() for model in fused_model.models],
    "config": cfg
}
torch.save(final_payload, final_model_path)
print(f"[OK] Saved unified final production artifact to: {final_model_path}")

# Reload Verification
loaded_payload = torch.load(final_model_path, map_location=device, weights_only=False)
reloaded = ProbabilityFusionModel([build_model(name).to(device) for name in loaded_payload["model_names"]], loaded_payload["fusion_weights"]).to(device)
reloaded.load_state_dict(loaded_payload["state_dict"])
reloaded.eval()

with torch.no_grad():
    sample_images, sample_targets = next(iter(test_loader))
    sample_output = reloaded(sample_images.to(device))
assert sample_output.shape[1] == len(class_names) and torch.isfinite(sample_output).all()
print(f"[OK] Reload verification PASSED! Sample batch output shape: {tuple(sample_output.shape)}")

# Reusable Interactive Inference Function
def predict_emotion_from_image(image_input, model_instance=reloaded):
    """Accepts image file path or PIL Image and outputs emotion predictions with visual confidence plot."""
    if isinstance(image_input, (str, Path)):
        img = Image.open(image_input).convert("RGB")
    elif isinstance(image_input, Image.Image):
        img = image_input.convert("RGB")
    else:
        raise TypeError("Expected filepath or PIL Image")
        
    t_tensor = eval_transform(img).unsqueeze(0).to(device)
    with torch.no_grad():
        probs = model_instance(t_tensor)[0].cpu().numpy()
    pred_idx = int(np.argmax(probs))
    pred_label = class_names[pred_idx]
    confidence = float(probs[pred_idx])
    
    fig, (ax_img, ax_bar) = plt.subplots(1, 2, figsize=(10, 4), gridspec_kw={'width_ratios': [1, 1.8]})
    ax_img.imshow(img)
    ax_img.set_title(f"Predicted: {pred_label.upper()}\n({confidence*100:.1f}%)", fontweight="bold", color="#1e3a8a")
    ax_img.axis("off")
    
    y_pos = np.arange(len(class_names))
    colors = ['#1e3a8a' if i == pred_idx else '#94a3b8' for i in range(len(class_names))]
    bars = ax_bar.barh(y_pos, probs * 100.0, color=colors)
    ax_bar.set_yticks(y_pos)
    ax_bar.set_yticklabels(class_names, fontweight="bold")
    ax_bar.set_xlabel("Confidence Probability (%)", fontweight="bold")
    ax_bar.set_xlim(0, 100)
    ax_bar.grid(axis="x", alpha=0.3)
    for bar in bars:
        w = bar.get_width()
        if w > 2:
            ax_bar.text(w + 1, bar.get_y() + bar.get_height()/2, f"{w:.1f}%", va="center", fontsize=9, fontweight="bold")
    plt.tight_layout()
    plt.show()
    plt.close()
    
    return {
        "predicted_emotion": pred_label,
        "confidence": confidence,
        "probabilities": dict(zip(class_names, map(float, probs)))
    }

demo_sample_path = test_records.iloc[0]["path"]
demo_result = predict_emotion_from_image(demo_sample_path)
print("[DEMO RESULT] Predicted Emotion:", demo_result["predicted_emotion"], f"(Confidence: {demo_result['confidence']*100:.1f}%)")


## Outputs & Artifacts Summary

The pipeline produces the following production artifacts in the configured output directory (`outputs/`):
- `fused_model.pt`: Single unified deployable PyTorch bundle containing all 3 fine-tuned model weights, metadata, optimal Dirichlet weights, and preprocessing parameters.
- `complete_model_comparison.csv`: Side-by-side benchmark table comparing individual models (`efficientnet_b2`, `resnet50`, `vit_b_16`) against the `fused_model`.
- `fused_model_confusion_matrix_percentage.png`: Dual percentage (%) and exact sample count confusion matrix on the 7,048 held-out test images.
- `all_models_training_validation_metrics.png`: Epoch-by-epoch loss, accuracy, and macro-F1 trajectories across all 3 models.
- `*_roc_curves.png`: Multiclass One-vs-Rest ROC curves with per-class AUC scores.
- `dataset_inventory.csv`: Complete verified audit trail of all 35,269 clean 7-class FER2013+ image files.

## 13. External Benchmark Evaluation: Testing Fused Model on FER2013 (7 Classes)

This section evaluates the fine-tuned **Fused Model** on the original **FER2013** benchmark dataset:
- **Dataset Path**: `C:\Users\Shahzaib Hussain\Desktop\Shahzaib Hussain\FYP\Datasets\Facial Dataset\FER2013`
- **Class Structure**: 7 emotion classes (`angry`, `disgust`, `fear`, `happy`, `neutral`, `sad`, `surprise`).
- **1-to-1 Taxonomy Alignment**: Because the fine-tuning pipeline is now standardized to the 7 basic Ekman emotions, the model's 7 output logits map directly **1-to-1** with the FER2013 ground truth classes.

### Evaluation Features:
1. **Direct Class Alignment**:
   Maps folder names seamlessly: `angry` $\leftrightarrow$ `anger`, `happy` $\leftrightarrow$ `happiness`, `sad` $\leftrightarrow$ `sadness`, etc.
2. **Dynamic Compatibility**:
   Automatically detects whether the loaded fused model is 7-class (direct evaluation) or 8-class (adaptive contempt re-normalization).
3. **Comprehensive Metrics & Reporting**:
   Computes Accuracy, Macro F1, Weighted F1, Classification Report, dual-annotated Confusion Matrix, and Per-Class Performance Bar Chart.
4. **Interactive Single-Image Tester**:
   Provides `predict_emotion_fer2013()` for real-time visual testing.

In [ ]:
# ==============================================================================
# 13. EXTERNAL BENCHMARK EVALUATION: FER2013 (7 CLASSES)
# ==============================================================================
import os
import glob
import time
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models as tv_models
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"[EVAL-SETUP] Using hardware device: {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")

# 1. Dataset Path Configuration
FER2013_ROOT = Path(cfg["data"].get("fer2013_benchmark_dir", r"C:\Users\Shahzaib Hussain\Desktop\Shahzaib Hussain\FYP\Datasets\Facial Dataset\FER2013"))
EVAL_SPLIT = "test"  # Options: 'test' (7,178 images) or 'train' (28,709 images)
FER2013_EVAL_DIR = FER2013_ROOT / EVAL_SPLIT
assert FER2013_EVAL_DIR.exists(), f"FER2013 directory not found at: {FER2013_EVAL_DIR}"

OUTPUT_DIR = Path("./outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 2. Class Definitions & Alignments
FER2013_CLASSES = ["angry", "disgust", "fear", "happy", "neutral", "sad", "surprise"]

FER2013_TO_MODEL_MAP = {
    "angry": "anger",
    "disgust": "disgust",
    "fear": "fear",
    "happy": "happiness",
    "neutral": "neutral",
    "sad": "sadness",
    "surprise": "surprise"
}

# 3. Load Pre-Trained Fused Model Bundle
final_model_path = OUTPUT_DIR / "fused_model.pt"
assert final_model_path.exists(), f"Missing production bundle at: {final_model_path}. Please run the pipeline cells above first."
payload = torch.load(final_model_path, map_location=device, weights_only=False)

model_class_names = payload.get("class_names", class_names)
num_model_classes = len(model_class_names)
print(f"[MODEL-INFO] Fused Model loaded from: {final_model_path}")
print(f"[MODEL-INFO] Output Classes ({num_model_classes}): {model_class_names}")

class ProbabilityFusionModel(nn.Module):
    def __init__(self, models, weights):
        super().__init__()
        self.models = nn.ModuleList(models)
        self.register_buffer("weights", torch.tensor(weights, dtype=torch.float32))
    def forward(self, x):
        probabilities = [torch.softmax(model(x), dim=1) for model in self.models]
        normalized_w = self.weights / self.weights.sum()
        return sum(w * p for w, p in zip(normalized_w, probabilities))

def build_model_arch(name, num_classes=num_model_classes):
    if name == "efficientnet_b2":
        model = tv_models.efficientnet_b2(weights=None)
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
    elif name == "resnet50":
        model = tv_models.resnet50(weights=None)
        model.fc = nn.Linear(model.fc.in_features, num_classes)
    elif name == "vit_b_16":
        model = tv_models.vit_b_16(weights=None)
        model.heads.head = nn.Linear(model.heads.head.in_features, num_classes)
    else:
        raise ValueError(f"Unknown architecture: {name}")
    return model

fused_eval_model = ProbabilityFusionModel(
    [build_model_arch(m).to(device) for m in payload["model_names"]],
    payload["fusion_weights"]
).to(device)
fused_eval_model.load_state_dict(payload["state_dict"])
fused_eval_model.eval()

# 4. Scan & Index FER2013 Dataset
model_class_to_idx = {c: i for i, c in enumerate(model_class_names)}
records = []
for c in FER2013_CLASSES:
    folder = FER2013_EVAL_DIR / c
    files = list(folder.glob("*.*"))
    target_name = FER2013_TO_MODEL_MAP[c]
    t_idx = model_class_to_idx.get(target_name, -1)
    for f in files:
        records.append({
            "path": str(f),
            "fer2013_label": c,
            "model_label": target_name,
            "target": t_idx
        })

fer2013_df = pd.DataFrame(records)
print(f"[DATASET-INVENTORY] Discovered {len(fer2013_df):,} images in FER2013 '{EVAL_SPLIT}' split:")
display(fer2013_df.groupby(["fer2013_label", "model_label"]).size().reset_index(name="sample_count"))

eval_transform_fer = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

class FERFastDataset(Dataset):
    def __init__(self, df, transform):
        self.paths = df["path"].to_list()
        self.targets = df["target"].to_numpy(dtype=np.int64)
        self.transform = transform
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert("RGB")
        return self.transform(img), int(self.targets[idx])

fer_loader = DataLoader(
    FERFastDataset(fer2013_df, eval_transform_fer),
    batch_size=64,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

# 5. Run GPU-Accelerated Batched Inference
all_raw_probs = []
all_targets = []

t_start = time.time()
with torch.no_grad():
    for imgs, tgts in fer_loader:
        probs = fused_eval_model(imgs.to(device)).cpu().numpy()
        all_raw_probs.append(probs)
        all_targets.append(tgts.numpy())

all_raw_probs = np.concatenate(all_raw_probs, axis=0)
all_targets = np.concatenate(all_targets, axis=0)
eval_duration = time.time() - t_start
fps = len(fer2013_df) / eval_duration
print(f"[INFERENCE] Evaluated {len(fer2013_df):,} images in {eval_duration:.2f}s ({fps:.1f} FPS)")

# 6. Evaluation Logic (Dynamic 7-Class vs 8-Class Handling)
if num_model_classes == 7:
    active_eval_names = model_class_names
    preds_eval = np.argmax(all_raw_probs, axis=1)
    targets_eval = all_targets
else:
    # Legacy 8-class handling: mask contempt
    contempt_idx = model_class_names.index("contempt")
    active_indices = [i for i in range(num_model_classes) if i != contempt_idx]
    active_eval_names = [model_class_names[i] for i in active_indices]
    idx_map = {orig: new for new, orig in enumerate(active_indices)}
    targets_eval = np.array([idx_map[t] for t in all_targets])
    probs_7 = all_raw_probs[:, active_indices]
    probs_7 = probs_7 / probs_7.sum(axis=1, keepdims=True)
    preds_eval = np.argmax(probs_7, axis=1)

acc = accuracy_score(targets_eval, preds_eval)
macro_f1 = f1_score(targets_eval, preds_eval, average="macro", zero_division=0)
micro_f1 = f1_score(targets_eval, preds_eval, average="micro", zero_division=0)
weighted_f1 = f1_score(targets_eval, preds_eval, average="weighted", zero_division=0)

print("=" * 75)
print(f"FER2013 BENCHMARK EVALUATION ({EVAL_SPLIT.upper()} SPLIT)")
print("=" * 75)
print(f"Dataset Path        : {FER2013_EVAL_DIR}")
print(f"Total Test Images   : {len(fer2013_df):,}")
print(f"Active Categories   : {active_eval_names}")
print(f"• Test Accuracy     : {acc * 100:.2f}%")
print(f"• Macro F1-Score    : {macro_f1 * 100:.2f}%")
print(f"• Weighted F1-Score : {weighted_f1 * 100:.2f}%")
print(f"• Micro F1-Score    : {micro_f1 * 100:.2f}%")
print("=" * 75)

clf_dict = classification_report(targets_eval, preds_eval, target_names=active_eval_names, output_dict=True, zero_division=0)
clf_df = pd.DataFrame(clf_dict).T
display(clf_df.round(4))
clf_df.to_csv(OUTPUT_DIR / "fer2013_7class_classification_report.csv")

eval_metrics_payload = {
    "dataset": f"FER2013 ({EVAL_SPLIT})",
    "dataset_path": str(FER2013_EVAL_DIR),
    "total_samples": len(fer2013_df),
    "accuracy": float(acc),
    "macro_f1": float(macro_f1),
    "weighted_f1": float(weighted_f1),
    "micro_f1": float(micro_f1),
    "per_class": {c: {k: float(v) for k, v in clf_dict[c].items()} for c in active_eval_names}
}
with open(OUTPUT_DIR / "fer2013_7class_metrics.json", "w") as f:
    json.dump(eval_metrics_payload, f, indent=2)

# 7. Dual-Annotated Percentage & Count Confusion Matrix
cm_ext = confusion_matrix(targets_eval, preds_eval)
cm_ext_norm = cm_ext.astype("float") / cm_ext.sum(axis=1)[:, np.newaxis]

annot_labels = np.empty_like(cm_ext, dtype=object)
for i in range(cm_ext.shape[0]):
    for j in range(cm_ext.shape[1]):
        annot_labels[i, j] = f"{cm_ext_norm[i, j]*100:.1f}%\n({cm_ext[i, j]})"

fig, ax = plt.subplots(figsize=(9, 8))
sns.heatmap(cm_ext_norm * 100.0, annot=annot_labels, fmt="", cmap="Blues", cbar=True,
            xticklabels=active_eval_names, yticklabels=active_eval_names, ax=ax, annot_kws={"fontsize": 10, "fontweight": "bold"})
ax.set_title(f"FER2013 7-Class Test Confusion Matrix\nFused Model | Accuracy: {acc*100:.2f}% | Weighted F1: {weighted_f1*100:.2f}%", fontsize=12, fontweight="bold", pad=14)
ax.set_xlabel("Predicted Emotion", fontweight="bold", fontsize=11)
ax.set_ylabel("True Emotion", fontweight="bold", fontsize=11)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "fer2013_7class_confusion_matrix.png", dpi=200)
plt.show()
plt.close()

# 8. Per-Class Accuracy & F1-Score Bar Chart
per_class_recalls = [cm_ext_norm[i, i] * 100.0 for i in range(len(active_eval_names))]
per_class_f1s = [clf_dict[c]["f1-score"] * 100.0 for c in active_eval_names]

x_idx = np.arange(len(active_eval_names))
bar_w = 0.35

fig, ax = plt.subplots(figsize=(11, 5))
rects1 = ax.bar(x_idx - bar_w/2, per_class_recalls, bar_w, label="Recall / Accuracy (%)", color="#1e3a8a")
rects2 = ax.bar(x_idx + bar_w/2, per_class_f1s, bar_w, label="F1-Score (%)", color="#0d9488")
ax.set_title("FER2013 (7 Classes) Per-Class Performance Breakdown", fontsize=13, fontweight="bold")
ax.set_xticks(x_idx)
ax.set_xticklabels(active_eval_names, fontweight="bold")
ax.set_ylabel("Percentage (%)", fontweight="bold")
ax.set_ylim(0, 105)
ax.legend(frameon=True)
ax.grid(axis="y", alpha=0.3)

for rect in rects1:
    h = rect.get_height()
    ax.text(rect.get_x() + rect.get_width()/2., h + 1.2, f"{h:.1f}%", ha="center", va="bottom", fontsize=9, fontweight="bold", color="#1e3a8a")
for rect in rects2:
    h = rect.get_height()
    ax.text(rect.get_x() + rect.get_width()/2., h + 1.2, f"{h:.1f}%", ha="center", va="bottom", fontsize=9, fontweight="bold", color="#0d9488")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "fer2013_7class_per_class_performance.png", dpi=200)
plt.show()
plt.close()

# 9. Interactive 7-Class Inference Function & Demo Sample Test
def predict_emotion_fer2013(image_input, model_instance=fused_eval_model):
    """Predicts emotion on a 7-class facial image with horizontal confidence bars."""
    if isinstance(image_input, (str, Path)):
        img = Image.open(image_input).convert("RGB")
    elif isinstance(image_input, Image.Image):
        img = image_input.convert("RGB")
    else:
        raise TypeError("Expected image filepath or PIL Image object.")
        
    t_tensor = eval_transform_fer(img).unsqueeze(0).to(device)
    with torch.no_grad():
        raw_probs = model_instance(t_tensor)[0].cpu().numpy()
        
    if num_model_classes == 7:
        probs = raw_probs
    else:
        probs = raw_probs[active_indices] / raw_probs[active_indices].sum()
        
    pred_idx = int(np.argmax(probs))
    pred_name = active_eval_names[pred_idx]
    confidence = float(probs[pred_idx])
    
    fig, (ax_img, ax_bar) = plt.subplots(1, 2, figsize=(10, 4), gridspec_kw={"width_ratios": [1, 1.8]})
    ax_img.imshow(img)
    ax_img.set_title(f"Predicted: {pred_name.upper()}\n({confidence*100:.1f}%)", fontweight="bold", color="#1e3a8a")
    ax_img.axis("off")
    
    y_pos = np.arange(len(active_eval_names))
    colors = ["#1e3a8a" if i == pred_idx else "#94a3b8" for i in range(len(active_eval_names))]
    bars = ax_bar.barh(y_pos, probs * 100.0, color=colors)
    ax_bar.set_yticks(y_pos)
    ax_bar.set_yticklabels(active_eval_names, fontweight="bold")
    ax_bar.set_xlabel("Confidence Probability (%)", fontweight="bold")
    ax_bar.set_xlim(0, 100)
    ax_bar.grid(axis="x", alpha=0.3)
    
    for bar in bars:
        w = bar.get_width()
        if w > 2:
            ax_bar.text(w + 1, bar.get_y() + bar.get_height()/2, f"{w:.1f}%", va="center", fontsize=9, fontweight="bold")
    plt.tight_layout()
    plt.show()
    plt.close()
    
    return {
        "predicted_emotion": pred_name,
        "confidence": confidence,
        "probabilities": dict(zip(active_eval_names, map(float, probs)))
    }

demo_row = fer2013_df.iloc[0]
print(f"[DEMO-INFERENCE] Testing Sample Image: {demo_row['path']}")
print(f"[DEMO-INFERENCE] Ground-Truth FER2013 Label: {demo_row['fer2013_label']} ({demo_row['model_label']})")
demo_res = predict_emotion_fer2013(demo_row["path"])
print(f"[DEMO-INFERENCE] Predicted Emotion: {demo_res['predicted_emotion']} (Confidence: {demo_res['confidence']*100:.1f}%)")
